# Surgical Phase Training Pipeline (Colab)

End-to-end: install deps, build manifest, train/eval CNN+LSTM, TCN, Transformer.

In [ ]:
!nvidia-smi || true
!python -V

In [ ]:
%cd /content
!git clone https://github.com/ericnerwala/surgical-phase-colab-starter.git
%cd /content/surgical-phase-colab-starter
!pip -q install -r requirements.txt
!pip -q install -e .

In [ ]:
import urllib.request, zipfile
from pathlib import Path
root = Path('/content/data')
root.mkdir(exist_ok=True)
zip_path = root / 'CholecT50-Challenge-Validation.zip'
if not zip_path.exists():
    urllib.request.urlretrieve('https://s3.unistra.fr/camma_public/datasets/cholect50/CholecT50-Challenge-Validation.zip', zip_path)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(root)
!python scripts/build_manifest.py --data-root /content/data/cholect50-challenge-val --val-videos VID73 --test-videos VID75

In [ ]:
# Train CNN+LSTM baseline
!python scripts/train.py --config configs/base.yaml
!python scripts/eval.py --config configs/base.yaml --checkpoint outputs/cnn_lstm_base/best.pt --split test --out outputs/cnn_lstm_base/test_metrics.json

In [ ]:
# Train TCN
!python scripts/train.py --config configs/tcn.yaml
!python scripts/eval.py --config configs/tcn.yaml --checkpoint outputs/tcn_base/best.pt --split test --out outputs/tcn_base/test_metrics.json

In [ ]:
# Train Transformer
!python scripts/train.py --config configs/transformer.yaml
!python scripts/eval.py --config configs/transformer.yaml --checkpoint outputs/transformer_base/best.pt --split test --out outputs/transformer_base/test_metrics.json